# Derm ReasonEdit — Phase 1 Workbook

Human feedback → human ground truth for the **top-1 diagnosis** editing experiment.

**Goal of the experiment:** the edit set is exactly the cases where the target model's top-1 was *wrong* (baseline top-1 acc = 0). After ReasonEdit builds a codebook from the human's corrected reasoning and retrieves it via dual embedding, re-querying the same image should surface the human's corrected answer → top-1 acc → 1.

**Scope switches:** `(user, model)`. Defaults set in `config.py`.

> Run this notebook with the `dermato_llama` conda env (Python 3.11, has matplotlib + PIL):
> `/scratch/jq2uw/conda_ex/conda_venv/dermato_llama/bin/python`
> (The `reasonedit` env works for the pure data-engineering code but has a broken matplotlib.)

In [1]:
import sys, os, json
from pathlib import Path
from collections import Counter

# Make the shared package importable
SHARED = Path("/scratch/jq2uw/MME/derm_vlm_edit/editing/shared")
sys.path.insert(0, str(SHARED))

import config as C
from data_engineering.parse_annotations import load_records, filter_records, reconstruct_reasoning
from data_engineering.normalize_dx import normalize
from data_engineering.build_edit_set import build
print('shared root :', SHARED)
print('annotations :', C.ANNOTATIONS_CSV, '— exists:', C.ANNOTATIONS_CSV.exists())
print('image root  :', C.IMAGE_ROOT, '— exists:', C.IMAGE_ROOT.exists())

package root: /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/derm_reasonedit
annotations : /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/derm_vlms/data/annotations_export.csv — exists: True
image root  : /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/derm_vlms/results/images — exists: True


## 1. Global profiling of the annotation export

In [2]:
recs = load_records(C.ANNOTATIONS_CSV)
print('total annotation rows:', len(recs))
print('models   :', Counter(r.model for r in recs).most_common())
print('users    :', Counter(r.login_id for r in recs).most_common())
print('complete :', Counter(r.marked_complete for r in recs).most_common())
print('diag_1_label (target models, complete):',
      Counter(r.diag_1_label for r in recs if r.model in C.TARGET_MODELS and r.marked_complete).most_common())

total annotation rows: 939
models   : [('DermatoLlama', 318), ('MedGemma', 311), ('GPT-5.3', 310)]
users    : [('achiou@stanford.edu', 303), ('lhunterellul', 303), ('yosephdalia513@gmail.com', 157), ('jesse.y.liu.md@gmail.com', 151), ('roxanad', 16), ('kaverikarhade', 4), ('info@danielyanesmd.com', 2), ('demo_28e5b436e2d19f57', 1), ('labarnes@stanford.edu', 1), ('test', 1)]
complete : [(True, 910), (False, 29)]
diag_1_label (target models, complete): [('', 506), ('incorrect', 99), ('correct', 5)]


## 2. Usable top-1 errors by (user, model)
Usable = target model, marked complete, `diag_1` labeled incorrect, and a human correction present.

In [3]:
usable = [r for r in filter_records(recs, top1_errors_only=True, require_correction=True)
          if r.model in C.TARGET_MODELS]
print('global usable top-1 errors:', len(usable))
for (u, m), n in Counter((r.login_id, r.model) for r in usable).most_common():
    print(f'  {n:3d}   {m:13s} {u}')

global usable top-1 errors: 99
   25   MedGemma      jesse.y.liu.md@gmail.com
   21   MedGemma      achiou@stanford.edu
   17   DermatoLlama  achiou@stanford.edu
   15   DermatoLlama  jesse.y.liu.md@gmail.com
   12   DermatoLlama  lhunterellul
    8   MedGemma      lhunterellul
    1   DermatoLlama  roxanad


## 3. Normalization coverage (free-text → y16)

In [4]:
nds = [normalize(r.human_top1_raw) for r in usable]
print('multi-dx corrections :', sum(nd.is_multi for nd in nds))
print('y16 unmapped         :', sum(nd.y16 is None for nd in nds))
print('empty rationale      :', sum(not reconstruct_reasoning(r.reasoning_lines) for r in usable))
print()
print('y16 distribution:')
for k, n in Counter(nd.y16 or '<unmapped>' for nd in nds).most_common():
    print(f'  {n:3d}  {k}')
print()
print('any unmapped raw strings:')
for r, nd in zip(usable, nds):
    if nd.y16 is None:
        print('  ', repr(r.human_top1_raw), '-> norm', repr(nd.norm))

multi-dx corrections : 10
y16 unmapped         : 0
empty rationale      : 0

y16 distribution:
   18  Melanocytic Nevus
   17  Seborrheic Keratosis
   16  Non-neoplastic
   10  Actinic Keratosis
    9  Melanocytic Lesion
    8  Squamous Cell Carcinoma
    7  Other Benign
    5  Melanoma
    3  Squamous Cell Carcinoma In Situ
    2  Dermatofibroma
    2  Basal Cell Carcinoma
    1  Fibrous Papule
    1  Other Malignant

any unmapped raw strings:


## 4. Build the edit set for the current (user, model)

In [5]:
USER  = C.DEFAULT_USER
MODEL = C.DEFAULT_MODEL
edits, summary = build(user=USER, model=MODEL, rationale_mode=C.RATIONALE_MODE, write=True)
print(json.dumps({k: v for k, v in summary.items() if not k.endswith('_counts')}, indent=2))
print('\nwrote:', C.edit_set_path(USER, MODEL))

{
  "user": "jesse.y.liu.md@gmail.com",
  "model": "DermatoLlama",
  "rationale_mode": "edited_only",
  "question": "What is the single most likely diagnosis for this lesion?",
  "n_edits": 15,
  "n_images_missing": 0,
  "n_multi_dx_top1": 0,
  "n_y16_unmapped": 0,
  "n_empty_rationale": 0
}

wrote: /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/derm_reasonedit/outputs/edit_set__jesse_y_liu_md_at_gmail_com__DermatoLlama.jsonl


## 5. Qualitative before/after cards
Model's (wrong) top-1 vs the human's corrected top-1 + reconstructed reasoning, with the image.

In [6]:
import textwrap
import matplotlib.pyplot as plt
from PIL import Image

def show_cards(edits, n=6):
    n = min(n, len(edits))
    fig, axes = plt.subplots(n, 2, figsize=(13, 3.1 * n), gridspec_kw={'width_ratios': [1, 2.4]})
    if n == 1:
        axes = axes.reshape(1, 2)
    for i, e in enumerate(edits[:n]):
        try:
            axes[i, 0].imshow(Image.open(e['image']).convert('RGB'))
        except Exception as exc:
            axes[i, 0].text(0.5, 0.5, f'no image\n{exc}', ha='center')
        axes[i, 0].axis('off')
        axes[i, 0].set_title(e['case_id'], fontsize=9)
        txt = (
            f"MODEL top-1 (wrong): {e['model_top1']}\n"
            f"HUMAN top-1 (raw)  : {e['human_top1_raw']}\n"
            f"HUMAN top-1 (norm) : {e['answer']}\n"
            f"HUMAN top-1 (y16)  : {e['human_top1_y16']}\n\n"
            f"human reasoning:\n{textwrap.fill(e['rationale'], 90)}"
        )
        axes[i, 1].axis('off')
        axes[i, 1].text(0.0, 1.0, txt, va='top', ha='left', fontsize=9, family='monospace')
    plt.tight_layout()
    plt.show()

show_cards(edits, n=6)

## 6. Sanity: one full ReasonEdit-schema record

In [7]:
print(json.dumps(edits[0], indent=2, ensure_ascii=False))

{
  "uid": "jesse_y_liu_md_at_gmail_com__DermatoLlama__1025_combined",
  "image": "/sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/derm_vlms/results/images/1025_combined.jpg",
  "question": "What is the single most likely diagnosis for this lesion?",
  "answer": "lichenoid keratosis",
  "rationale": "erythematous nonspecific macule",
  "cot": "erythematous nonspecific macule",
  "login_id": "jesse.y.liu.md@gmail.com",
  "model": "DermatoLlama",
  "case_id": "1025_combined",
  "model_top1": "Basal Cell Carcinoma",
  "human_top1_raw": "Lichenoid keratosis",
  "human_top1_y16": "Seborrheic Keratosis",
  "human_top1_is_multi": false,
  "diagnosis_order": "[3, 2, 1]",
  "n_reasoning_lines": 3,
  "n_human_edited_lines": 1,
  "image_exists": true
}
